In [ ]:
import os
from getpass import getpass


from roboflow import Roboflow
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: "))
project = rf.workspace("hoainam-pusiu").project("nhan-dien-rac-dien-tu-ehsyl")
version = project.version(21)
dataset = version.download("yolo26")



In [ ]:
import os
import cv2
dataset_path = dataset.location
print("Dataset path:", dataset_path)

# ===== 2. Khai báo class (đọc từ file data.yaml nếu có) =====
classes = []

yaml_path = os.path.join(dataset_path, "data.yaml")
if os.path.exists(yaml_path):
    import yaml
    with open(yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)
        classes = data_yaml['names']

print("Classes:", classes)

# ===== 3. Tạo thư mục CNN =====
cnn_root = os.path.join(dataset_path, "cnn_dataset")

splits = ["train", "valid", "test"]

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(cnn_root, split, cls), exist_ok=True)

# ===== 4. Hàm convert YOLO → pixel =====
def yolo_to_pixel(x_c, y_c, w, h, img_w, img_h):
    x_min = int((x_c - w/2) * img_w)
    y_min = int((y_c - h/2) * img_h)
    x_max = int((x_c + w/2) * img_w)
    y_max = int((y_c + h/2) * img_h)

    # fix tràn biên
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(img_w, x_max)
    y_max = min(img_h, y_max)

    return x_min, y_min, x_max, y_max

# ===== 5. Crop ảnh =====
for split in splits:
    img_dir = os.path.join(dataset_path, split, "images")
    label_dir = os.path.join(dataset_path, split, "labels")

    if not os.path.exists(img_dir):
        print(f"Không có {split}")
        continue

    for img_name in os.listdir(img_dir):
        img_path = os.path.join(img_dir, img_name)
        label_path = os.path.join(label_dir, img_name.replace(".jpg", ".txt"))

        if not os.path.exists(label_path):
            continue

        image = cv2.imread(img_path)
        if image is None:
            continue

        img_h, img_w = image.shape[:2]

        with open(label_path, "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split()
            class_id = int(parts[0])

            if len(parts) == 5: # Standard YOLO bounding box format
                x_c, y_c, w, h = map(float, parts[1:])
            else: # YOLO segmentation format, convert polygon to bounding box
                coords = list(map(float, parts[1:]))
                xs = coords[0::2] # All x coordinates
                ys = coords[1::2] # All y coordinates

                min_x = min(xs)
                max_x = max(xs)
                min_y = min(ys)
                max_y = max(ys)

                # Convert to YOLO center_x, center_y, width, height format
                w = max_x - min_x
                h = max_y - min_y
                x_c = min_x + w / 2
                y_c = min_y + h / 2

            x_min, y_min, x_max, y_max = yolo_to_pixel(x_c, y_c, w, h, img_w, img_h)

            crop = image[y_min:y_max, x_min:x_max]

            # bỏ crop lỗi
            if crop.size == 0:
                print(f"[DEBUG] Crop failed for {img_name}, label {i}:")
                print(f"  Original image size: W={img_w}, H={img_h}")
                print(f"  Calculated crop coordinates: x_min={x_min}, y_min={y_min}, x_max={x_max}, y_max={y_max}")
                print(f"  YOLO box: x_c={x_c}, y_c={y_c}, w={w}, h={h}")
                continue

            class_name = classes[class_id]

            save_path = os.path.join(
                cnn_root,
                split,
                class_name,
                f"{img_name.replace('.jpg','')}_{i}.jpg"
            )

            cv2.imwrite(save_path, crop)

print("✅ Đã tạo xong dataset CNN!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import os

# --- 1. Hyperparameters & Configuration ---
class ResearchConfig:
    DATA_DIR = '/content/nhận-diện-rác-điện-tử-16/cnn_dataset'
    BATCH_SIZE = 32
    EPOCHS = 30
    LR = 2e-5  # Thường thấp hơn cho Transfer Learning
    WEIGHT_DECAY = 0.05
    NUM_CLASSES = 7
    IMG_SIZE = 224
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 2. Advanced Data Augmentation ---
train_transform = transforms.Compose([
    transforms.Resize((ResearchConfig.IMG_SIZE, ResearchConfig.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((ResearchConfig.IMG_SIZE, ResearchConfig.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Data Loaders ---
train_set = datasets.ImageFolder(os.path.join(ResearchConfig.DATA_DIR, 'train'), train_transform)
val_set = datasets.ImageFolder(os.path.join(ResearchConfig.DATA_DIR, 'valid'), val_transform)

train_loader = DataLoader(train_set, batch_size=ResearchConfig.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=ResearchConfig.BATCH_SIZE, shuffle=False, num_workers=2)

# --- 4. Model Definition (ConvNeXt Tiny) ---
def build_convnext():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    # Thay đổi đầu ra phù hợp với số lớp
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, ResearchConfig.NUM_CLASSES)
    return model.to(ResearchConfig.DEVICE)

model = build_convnext()

# --- 5. Loss, Optimizer, Scheduler ---
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # Label smoothing chuẩn NCKH
optimizer = optim.AdamW(model.parameters(), lr=ResearchConfig.LR, weight_decay=ResearchConfig.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ResearchConfig.EPOCHS)
scaler = GradScaler() # Cho Mixed Precision Training

# --- 6. Training Loop ---
best_acc = 0.0
for epoch in range(ResearchConfig.EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{ResearchConfig.EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(ResearchConfig.DEVICE), labels.to(ResearchConfig.DEVICE)

        optimizer.zero_grad()

        with autocast(): # Mixed Precision
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / len(train_set)

    # Validation
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(ResearchConfig.DEVICE), labels.to(ResearchConfig.DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / len(val_set)
    print(f'Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_convnext_research.pth')
        print(f'==> Saved Best Model with Acc: {best_acc:.2f}%')

    scheduler.step()

In [ ]:
no_detection_count = sum(1 for s in misclassified_samples if s['pred'] == 'No Detection')
wrong_class_count = len(misclassified_samples) - no_detection_count

print(f'--- PHÂN TÍCH LỖI CHI TIẾT ---')
print(f'Tổng số ảnh lỗi: {len(misclassified_samples)}')
print(f'Số mẫu YOLO không tìm thấy (No Detection): {no_detection_count} ({(no_detection_count/470)*100:.2f}% tổng tập test)')
print(f'Số mẫu bị phân loại sai (Wrong Class): {wrong_class_count}')

if no_detection_count > 0:
    print('\n=> Nhận xét: Tỷ lệ bỏ sót vật thể của YOLO là nguyên nhân chính gây giảm Recall.')

# kiểm tra cnn

## convnext

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# --- 1. Cấu hình kiểm tra ---
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test'
MODEL_PATH = '/content/drive/MyDrive/best_convnext_research5.pth'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
NUM_CLASSES = 7

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Khởi tạo Model & Load Weights ---
def load_research_model(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)

    state_dict = torch.load(path, map_location=DEVICE)
    # Xử lý nếu model được lưu từ torch.compile
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE)

# --- 3. Load Dữ liệu ---
test_set = datasets.ImageFolder(TEST_DIR, test_transform)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
class_names = test_set.classes

model = load_research_model(MODEL_PATH, NUM_CLASSES)
model.eval()

# --- 4. Chạy Inference ---
all_preds = []
all_labels = []

print(f"--- Đang đánh giá mô hình: {MODEL_PATH} ---")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- 5. Xuất báo cáo ---
print("\n[BÁO CÁO PHÂN LOẠI CHI TIẾT]")
print(classification_report(all_labels, all_preds, target_names=class_names, digits = 4))

# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thực tế')
plt.title('Confusion Matrix - ConvNeXt Research Model')
plt.show()

In [ ]:
import time
import torch

# Đảm bảo model đã được load (sử dụng biến model từ các cell trước)
# Nếu chưa, bạn cần chạy cell định nghĩa model trước.

def measure_inference_time(model, img_size=224, num_repeats=100):
    model.eval()
    dummy_input = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    # Warm-up (chạy nháp để khởi động GPU)
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    # Đo thời gian
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    timings = np.zeros((num_repeats, 1))

    with torch.no_grad():
        for rep in range(num_repeats):
            starter.record()
            _ = model(dummy_input)
            ender.record()
            torch.cuda.synchronize()
            curr_time = starter.elapsed_time(ender)
            timings[rep] = curr_time

    avg_time = np.sum(timings) / num_repeats
    fps = 1000 / avg_time

    print(f'--- THÔNG SỐ SUY LUẬN CONVNEXT ---')
    print(f'Thời gian trung bình: {avg_time:.2f} ms / ảnh')
    print(f'Tốc độ xử lý (FPS): {fps:.2f} frames per second')

measure_inference_time(model)

## eff đánh giá

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# --- 1. Cấu hình kiểm tra EfficientNetV2 ---
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test'
MODEL_PATH_EFF = '/content/drive/MyDrive/best_efficientnet_v2_research.pth'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
NUM_CLASSES = 7

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Khởi tạo Model & Load Weights ---
def load_effnet_research(path, num_classes):
    model = models.efficientnet_v2_s(weights=None)
    n_inputs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(n_inputs, num_classes)

    state_dict = torch.load(path, map_location=DEVICE)
    # Xử lý prefix nếu cần
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE)

# --- 3. Load Dữ liệu ---
test_set = datasets.ImageFolder(TEST_DIR, test_transform)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
class_names = test_set.classes

model_eff_eval = load_effnet_research(MODEL_PATH_EFF, NUM_CLASSES)
model_eff_eval.eval()

# --- 4. Chạy Inference ---
all_preds = []
all_labels = []

print(f"--- Đang đánh giá mô hình EfficientNetV2: {MODEL_PATH_EFF} ---")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model_eff_eval(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- 5. Xuất báo cáo ---
print("\n[BÁO CÁO PHÂN LOẠI CHI TIẾT - EFFICIENTNETV2]")
print(classification_report(all_labels, all_preds, target_names=class_names, digits = 4))

# Vẽ Confusion Matrix
cm_eff = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_eff, annot=True, fmt='d', cmap='YlGnBu', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thực tế')
plt.title('Confusion Matrix - EfficientNetV2 Research Model')
plt.show()

In [ ]:
import time
import torch
import numpy as np

def measure_effnet_inference(model, img_size=224, num_repeats=100):
    model.eval()
    # Sử dụng biến DEVICE và model_eff_eval đã được định nghĩa
    dummy_input = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    # Warm-up
    print("--- Đang khởi động GPU (Warm-up) ---")
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    # Đo thời gian bằng CUDA Events
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    timings = np.zeros((num_repeats, 1))

    print(f"--- Đang đo thời gian suy luận trên {num_repeats} lần thử ---")
    with torch.no_grad():
        for rep in range(num_repeats):
            starter.record()
            _ = model(dummy_input)
            ender.record()
            torch.cuda.synchronize()
            curr_time = starter.elapsed_time(ender)
            timings[rep] = curr_time

    avg_time = np.sum(timings) / num_repeats
    fps = 1000 / avg_time

    print(f'\n--- THÔNG SỐ SUY LUẬN EFFICIENTNETV2 ---')
    print(f'Thời gian trung bình: {avg_time:.2f} ms / ảnh')
    print(f'Tốc độ xử lý (FPS): {fps:.2f} frames per second')

# Gọi hàm đo
measure_effnet_inference(model_eff_eval)

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

# --- 1. Cấu hình kiểm tra trên tập VALID ---
VALID_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/valid'
MODEL_PATH_EFF = '/content/drive/MyDrive/best_efficientnet_v2_research.pth'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
NUM_CLASSES = 7

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Load Dữ liệu Valid ---
valid_set = datasets.ImageFolder(VALID_DIR, val_transform)
valid_loader = DataLoader(valid_set, batch_size=32, shuffle=False)
class_names = valid_set.classes

# --- 3. Load Model ---
def load_model(path, num_classes):
    model = models.efficientnet_v2_s(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    state_dict = torch.load(path, map_location=DEVICE)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE)

model_eff_val = load_model(MODEL_PATH_EFF, NUM_CLASSES)
model_eff_val.eval()

# --- 4. Chạy Inference ---
val_preds = []
val_labels = []

with torch.no_grad():
    for inputs, labels in valid_loader:
        inputs = inputs.to(DEVICE)
        outputs = model_eff_val(inputs)
        _, preds = torch.max(outputs, 1)
        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.numpy())

# --- 5. Xuất báo cáo 4 chữ số thập phân ---
print(f"--- BÁO CÁO PHÂN LOẠI TRÊN TẬP VALID (4 CHỮ SỐ) ---")
print(classification_report(val_labels, val_preds, target_names=class_names, digits=4))

In [ ]:
print(classification_report(all_labels, all_preds, target_names=class_names, digits = 4))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import os

# --- 1. Hyperparameters & Configuration ---
class EfficientConfig:
    DATA_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset'
    BATCH_SIZE = 32
    EPOCHS = 40
    LR = 1e-4
    WEIGHT_DECAY = 1e-5
    NUM_CLASSES = 7
    IMG_SIZE = 224
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 2. Data Transforms ---
train_transform = transforms.Compose([
    transforms.Resize((EfficientConfig.IMG_SIZE, EfficientConfig.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((EfficientConfig.IMG_SIZE, EfficientConfig.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Data Loaders ---
train_set = datasets.ImageFolder(os.path.join(EfficientConfig.DATA_DIR, 'train'), train_transform)
val_set = datasets.ImageFolder(os.path.join(EfficientConfig.DATA_DIR, 'valid'), val_transform)

train_loader = DataLoader(train_set, batch_size=EfficientConfig.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=EfficientConfig.BATCH_SIZE, shuffle=False, num_workers=2)

# --- 4. Model Definition (EfficientNetV2-S) ---
def build_efficientnet():
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    n_inputs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(n_inputs, EfficientConfig.NUM_CLASSES)
    return model.to(EfficientConfig.DEVICE)

model_eff = build_efficientnet()

# --- 5. Loss, Optimizer, Scheduler ---
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model_eff.parameters(), lr=EfficientConfig.LR, weight_decay=EfficientConfig.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EfficientConfig.EPOCHS)
scaler = GradScaler()

# --- 6. Training Loop ---
best_val_acc = 0.0
for epoch in range(EfficientConfig.EPOCHS):
    model_eff.train()
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f'EffNet Epoch {epoch+1}/{EfficientConfig.EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(EfficientConfig.DEVICE), labels.to(EfficientConfig.DEVICE)

        optimizer.zero_grad()
        with autocast():
            outputs = model_eff(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total

    # Validation
    model_eff.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(EfficientConfig.DEVICE), labels.to(EfficientConfig.DEVICE)
            outputs = model_eff(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    print(f'EffNet Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_eff.state_dict(), 'best_efficientnet_v2_research.pth')
        print(f'==> Saved Best EfficientNet Model: {best_val_acc:.2f}%')

    scheduler.step()

## resnet101

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# --- 1. Cấu hình kiểm tra ResNet101 ---
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test'
MODEL_PATH_RESNET = '/content/drive/MyDrive/best_resnet101_research.pth'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
NUM_CLASSES = 7

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Khởi tạo Model & Load Weights ---
def load_resnet_research(path, num_classes):
    model = models.resnet101(weights=None)
    n_inputs = model.fc.in_features
    model.fc = nn.Linear(n_inputs, num_classes)

    state_dict = torch.load(path, map_location=DEVICE)
    # Xử lý prefix nếu model được lưu từ torch.compile (_orig_mod.)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE)

# --- 3. Load Dữ liệu ---
test_set = datasets.ImageFolder(TEST_DIR, test_transform)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
class_names = test_set.classes

model_resnet = load_resnet_research(MODEL_PATH_RESNET, NUM_CLASSES)
model_resnet.eval()

# --- 4. Chạy Inference ---
all_preds = []
all_labels = []

print(f"--- Đang đánh giá mô hình ResNet101: {MODEL_PATH_RESNET} ---")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model_resnet(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- 5. Xuất báo cáo ---
print("\n[BÁO CÁO PHÂN LOẠI CHI TIẾT - RESNET101]")
print(classification_report(all_labels, all_preds, target_names=class_names, digits = 4))

# Vẽ Confusion Matrix
cm_resnet = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Oranges', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thực tế')
plt.title('Confusion Matrix - ResNet101 Research Model')
plt.show()

In [ ]:
import time
import torch
import numpy as np

def measure_resnet_inference(model, img_size=224, num_repeats=100):
    model.eval()
    # Sử dụng biến DEVICE đã định nghĩa ở các cell trước
    dummy_input = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    # Warm-up
    print("--- Đang khởi động GPU (Warm-up) ---")
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    # Đo thời gian bằng CUDA Events để đảm bảo độ chính xác cao trên GPU
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    timings = np.zeros((num_repeats, 1))

    print(f"--- Đang đo thời gian suy luận trên {num_repeats} lần thử ---")
    with torch.no_grad():
        for rep in range(num_repeats):
            starter.record()
            _ = model(dummy_input)
            ender.record()
            torch.cuda.synchronize()
            curr_time = starter.elapsed_time(ender)
            timings[rep] = curr_time

    avg_time = np.sum(timings) / num_repeats
    fps = 1000 / avg_time

    print(f'\n--- THÔNG SỐ SUY LUẬN RESNET101 ---')
    print(f'Thời gian trung bình: {avg_time:.2f} ms / ảnh')
    print(f'Tốc độ xử lý (FPS): {fps:.2f} frames per second')

# Gọi hàm đo với model_resnet đã load ở cell trên
measure_resnet_inference(model_resnet)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Dữ liệu trích xuất từ các báo cáo classification_report đã chạy
models_names = ['ConvNeXt', 'ResNet101', 'EfficientNetV2']
accuracy = [0.9816, 0.9961, 0.9803]  # Accuracy từ tập test
f1_scores = [0.9800, 0.9960, 0.9786] # Macro Avg F1-score

x = np.arange(len(models_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='#3498db')
rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score (Macro)', color='#e74c3c')

ax.set_ylabel('Score')
ax.set_title('So sánh Accuracy và F1-Score giữa các mô hình')
ax.set_xticks(x)
ax.set_xticklabels(models_names)
ax.set_ylim(0.95, 1.0) # Zoom vào khoảng 0.95 - 1.0 để thấy sự khác biệt
ax.legend()

# Hàm thêm nhãn số trên đầu cột
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

fig.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# test yolo classfication

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best5.pt'

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names

# --- 3. Mapping class ---
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)

y_true = []
y_pred = []

print(f"--- Đang đánh giá YOLO trên tập CNN Test ({TEST_DIR}) ---")

# --- 4. Chạy Evaluation ---
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    if not os.path.isdir(class_path): continue

    # Sửa lỗi Syntax ở đây
    img_list = os.listdir(class_path)
    for img_name in img_list:
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            best_box_idx = boxes.conf.argmax()
            cls_yolo = int(boxes[best_box_idx].cls[0])
            pred_label_name = yolo_names[cls_yolo]
            final_pred = name_to_idx.get(pred_label_name, -1)
        else:
            final_pred = -1

        y_true.append(label_idx)
        y_pred.append(final_pred)

# --- 5. Báo cáo ---
print("\n[BÁO CÁO PHÂN LOẠI YOLO TRÊN TẬP CNN]")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

# --- 6. Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Dự đoán (YOLO)')
plt.ylabel('Thực tế')
plt.title('YOLO Confusion Matrix on CNN Dataset')
plt.show()

# test yolo cnn

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test'  # Đường dẫn tập test CNN
YOLO_MODEL_PATH = '/content/drive/MyDrive/best5.pt'
CNN_MODEL_PATH = 'best_convnext_research.pth'
IMG_SIZE = 224
NUM_CLASSES = 7

# --- 2. Load Models ---
# Load YOLO
yolo_model = YOLO(YOLO_MODEL_PATH)

# Load CNN (ConvNeXt Tiny)
def load_cnn_model(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)
    state_dict = torch.load(path, map_location=DEVICE)
    # Xử lý prefix nếu model được compile
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    model.to(DEVICE)
    model.eval()
    return model

cnn_model = load_cnn_model(CNN_MODEL_PATH, NUM_CLASSES)

# --- 3. Transforms ---
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Ensure class_names, yolo_names, and name_to_idx are defined before evaluation
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}
yolo_names = yolo_model.names # Make sure yolo_model is loaded before this

# --- 4. Chạy Evaluation ---
y_true = []
y_pred = []

print(f"--- Bắt đầu đánh giá Fusion trên: {len(class_names)} lớp ---")

for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    if not os.path.isdir(class_path): continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        # 4.1 YOLO Inference
        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        final_pred = -1  # mặc định là không dự đoán

        if len(boxes) > 0:
            # Lấy box có confidence cao nhất
            best_box_idx = boxes.conf.argmax()
            box = boxes[best_box_idx]
            conf = float(box.conf[0])
            cls_yolo = int(box.cls[0])
            yolo_label_name = yolo_names[cls_yolo]

            # Map tên lớp YOLO sang index của CNN dataset
            yolo_mapped_idx = name_to_idx.get(yolo_label_name, -1)

            if conf >= 0.9 and yolo_mapped_idx != -1:
                # YOLO tin cậy cao
                final_pred = yolo_mapped_idx
            else:
                # YOLO tin cậy thấp -> dùng CNN trên crop
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # Clamp tránh lỗi
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                crop = frame[y1:y2, x1:x2]

                if crop.size > 0:
                    img_input = transform(crop).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        outputs = cnn_model(img_input)
                        _, pred_cnn = torch.max(outputs, 1)
                    final_pred = pred_cnn.item()
                else:
                    final_pred = -1

        else:
            # Nếu YOLO không detect được gì, dùng CNN trên toàn bộ ảnh
            img_input = transform(frame).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                outputs = cnn_model(img_input)
                _, pred_cnn = torch.max(outputs, 1)
            final_pred = pred_cnn.item()

        y_true.append(label_idx)
        y_pred.append(final_pred)

# --- 5. Kết quả ---
print("\n[BÁO CÁO KẾT QUẢ FUSION (YOLO + CNN)]")
print(classification_report(y_true, y_pred, target_names=class_names, labels=list(range(NUM_CLASSES)), zero_division=0))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Fusion Confusion Matrix')
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-21/cnn_dataset/test' # Corrected path
YOLO_MODEL_PATH = '/content/drive/MyDrive/best5.pt'

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names

# --- 3. Mapping class ---
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)

y_true = []
y_pred = []

print(f"--- Đánh giá YOLO ONLY (Bao gồm các mẫu không detect được) ---")

# --- 4. Evaluation ---
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    if not os.path.isdir(class_path): continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        final_pred = -1
        if len(boxes) > 0:
            best_box_idx = boxes.conf.argmax()
            cls_yolo = int(boxes[best_box_idx].cls[0])
            final_pred = name_to_idx.get(yolo_names[cls_yolo], -1)
        else:
            final_pred = -1 # Không detect được

        y_true.append(label_idx)
        y_pred.append(final_pred)

# --- 5. Báo cáo ---
print("\n[BÁO CÁO YOLO ONLY]")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

# --- 6. Confusion Matrix ---
# Sử dụng labels=list(range(NUM_CLASSES)) để ma trận chỉ tập trung vào các lớp đã định nghĩa.
# Các dự đoán là -1 sẽ được tính là lỗi cho lớp thực tế tương ứng.
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Dự đoán (YOLO)')
plt.ylabel('Thực tế')
plt.title('YOLO Confusion Matrix (Tất cả mẫu)')
plt.show()

In [ ]:
from ultralytics import YOLO
import numpy as np

# 1. Load model
MODEL_PATH = '/content/drive/MyDrive/best5.pt'
data_yaml = '/content/nhận-diện-rác-điện-tử-21/data.yaml'
model = YOLO(MODEL_PATH)

# 2. Chạy validation trên tập test
metrics = model.val(data=data_yaml, split='test', imgsz=640, device=0)

# 3. Trích xuất chỉ số
# Lưu ý: Accuracy trong YOLO thường được xem xét qua mAP hoặc tỉ lệ phân loại đúng trên các box đã detect
precision = metrics.box.mp
recall = metrics.box.mr
f1_score = metrics.box.f1.mean()
map50 = metrics.box.map50

print(f"\n--- KẾT QUẢ PHÂN LOẠI SƠ BỘ YOLO26s ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1_score:.4f}")
print(f"mAP@0.5:   {map50:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import glob

# Lấy ngẫu nhiên vài ảnh từ tập test YOLO để hiển thị
test_images_path = '/content/nhận-diện-rác-điện-tử-21/test/images'
img_list = glob.glob(os.path.join(test_images_path, '*.jpg'))[:3]

plt.figure(figsize=(20, 10))

for i, img_path in enumerate(img_list):
    results = model.predict(source=img_path, conf=0.25, device=0, verbose=False)
    res_plotted = results[0].plot()

    plt.subplot(1, 3, i + 1)
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection Sample {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 1. Import
# =========================
from ultralytics import YOLO

# =========================
# 2. Đường dẫn (CẬP NHẬT ĐÚNG THƯ MỤC -16)
# =========================
MODEL_PATH = '/content/drive/MyDrive/best3.pt'
DATA_YAML = '/content/nhận-diện-rác-điện-tử-16/data.yaml'

# =========================
# 3. Load model
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 4. Đánh giá trên tập TEST
# =========================
metrics = model.val(
    data=DATA_YAML,
    split='test',     # Đảm bảo split này tồn tại trong data.yaml
    imgsz=640,
    conf=0.001,
    iou=0.6,
    device=0
)

# =========================
# 5. In kết quả
# =========================
print("\n===== YOLO EVALUATION ON TEST SET =====")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
print(metrics)

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Configuration ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Use the original YOLO test set
YOLO_TEST_IMG_DIR = '/content/nhận-diện-rác-điện-tử-16/test/images'
YOLO_TEST_LABEL_DIR = '/content/nhận-diện-rác-điện-tử-16/test/labels'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best1.pt'

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names
class_names = [yolo_names[i] for i in range(len(yolo_names))]
NUM_CLASSES = len(class_names)

y_true = []
y_pred = []

print(f"--- Evaluating YOLO on Original Test Set ({len(os.listdir(YOLO_TEST_IMG_DIR))} images) ---")

# --- 3. Evaluation Loop ---
for img_name in os.listdir(YOLO_TEST_IMG_DIR):
    img_path = os.path.join(YOLO_TEST_IMG_DIR, img_name)
    label_path = os.path.join(YOLO_TEST_LABEL_DIR, img_name.replace('.jpg', '.txt'))

    if not os.path.exists(label_path):
        continue

    frame = cv2.imread(img_path)
    if frame is None: continue

    # Get Ground Truth (Taking the first object in label file for simplified classification comparison)
    with open(label_path, 'r') as f:
        lines = f.readlines()
        if not lines: continue
        gt_class = int(lines[0].split()[0])

    # YOLO Inference
    results = yolo_model(frame, verbose=False, device=DEVICE)
    boxes = results[0].boxes

    if len(boxes) > 0:
        # Pick detection with highest confidence
        best_box_idx = boxes.conf.argmax()
        pred_class = int(boxes[best_box_idx].cls[0])
    else:
        pred_class = -1 # No detection

    y_true.append(gt_class)
    y_pred.append(pred_class)

# --- 4. Results Reporting ---
print("\n[YOLO ORIGINAL DATASET REPORT]")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted (YOLO)')
plt.ylabel('Actual (Ground Truth)')
plt.title('YOLO Confusion Matrix on Full Images')
plt.show()

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
YOLO_TEST_IMG_DIR = '/content/nhận-diện-rác-điện-tử-15/test/images'
YOLO_TEST_LABEL_DIR = '/content/nhận-diện-rác-điện-tử-15/test/labels'

YOLO_MODEL_PATH = '/content/drive/MyDrive/best4.pt'
CNN_MODEL_PATH = '/content/drive/MyDrive/best_convnext_research3.pth'

IMG_SIZE = 224
NUM_CLASSES = 7

# --- 2. Load Models ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names
class_names = [yolo_names[i] for i in range(len(yolo_names))]

def load_cnn_model(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)
    state_dict = torch.load(path, map_location=DEVICE)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    model.to(DEVICE)
    model.eval()
    return model

cnn_model = load_cnn_model(CNN_MODEL_PATH, NUM_CLASSES)

# --- 3. Transforms ---
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 4. Chạy Evaluation ---
y_true = []
y_pred = []

print(f"--- Bắt đầu đánh giá FUSION (YOLO + Rule Laptop + CNN) ---")

for img_name in os.listdir(YOLO_TEST_IMG_DIR):
    img_path = os.path.join(YOLO_TEST_IMG_DIR, img_name)
    label_path = os.path.join(YOLO_TEST_LABEL_DIR, img_name.replace('.jpg', '.txt'))

    if not os.path.exists(label_path): continue
    frame = cv2.imread(img_path)
    if frame is None: continue

    with open(label_path, 'r') as f:
        lines = f.readlines()
        if not lines: continue
        gt_class = int(lines[0].split()[0])

    results = yolo_model(frame, verbose=False, device=DEVICE)
    boxes = results[0].boxes
    final_pred = -1

    if len(boxes) > 0:
        best_box_idx = boxes.conf.argmax()
        box = boxes[best_box_idx]
        conf = float(box.conf[0])
        cls_yolo = int(box.cls[0])
        yolo_label = yolo_names[cls_yolo]

        # --- Logic Fusion Cải tiến ---
        # Nếu là laptop và tin cậy < 0.7 HOẶC tin cậy chung < 0.8
        if (yolo_label == 'keyboard' and conf < 0.9) or (conf < 0.6):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            h_img, w_img, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img, x2), min(h_img, y2)
            crop = frame[y1:y2, x1:x2]

            if crop.size > 0:
                img_input = transform(crop).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    outputs = cnn_model(img_input)
                    _, pred_cnn = torch.max(outputs, 1)
                final_pred = pred_cnn.item()
            else:
                final_pred = cls_yolo
        else:
            final_pred = cls_yolo
    else:
        final_pred = -1

    y_true.append(gt_class)
    y_pred.append(final_pred)

print("\n[BÁO CÁO KẾT QUẢ FUSION TỐI ƯU]")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# --- Thu thập các ảnh bị sai ---
misclassified_samples = []

print("--- Đang tìm kiếm các ảnh bị nhận diện sai... ---")
# Sử dụng lại logic từ cell 8e2fb273 để lấy danh sách ảnh sai
for img_name in os.listdir(YOLO_TEST_IMG_DIR):
    img_path = os.path.join(YOLO_TEST_IMG_DIR, img_name)
    label_path = os.path.join(YOLO_TEST_LABEL_DIR, img_name.replace('.jpg', '.txt'))
    if not os.path.exists(label_path): continue

    frame = cv2.imread(img_path)
    if frame is None: continue

    with open(label_path, 'r') as f:
        lines = f.readlines()
        if not lines: continue
        gt_class = int(lines[0].split()[0])

    results = yolo_model(frame, verbose=False, device=DEVICE)
    boxes = results[0].boxes
    final_pred = -1

    if len(boxes) > 0:
        best_box_idx = boxes.conf.argmax()
        box = boxes[best_box_idx]
        conf = float(box.conf[0])
        cls_yolo = int(box.cls[0])

        if conf >= 0.8:
            final_pred = cls_yolo
        else:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            h_img, w_img, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img, x2), min(h_img, y2)
            crop = frame[y1:y2, x1:x2]
            if crop.size > 0:
                img_input = transform(crop).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    outputs = cnn_model(img_input)
                    _, pred_cnn = torch.max(outputs, 1)
                final_pred = pred_cnn.item()
            else:
                final_pred = cls_yolo
    else:
        final_pred = -1

    if final_pred != gt_class:
        misclassified_samples.append({
            'img_path': img_path,
            'true': class_names[gt_class] if gt_class != -1 else "None",
            'pred': class_names[final_pred] if final_pred != -1 else "No Detection"
        })

# --- Hiển thị kết quả ---
num_to_show = min(12, len(misclassified_samples))
if num_to_show > 0:
    cols = 4
    rows = (num_to_show + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))

    for i in range(num_to_show):
        sample = misclassified_samples[i]
        img = cv2.imread(sample['img_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f"True: {sample['true']}\nPred: {sample['pred']}", color='red')
        plt.axis('off')

    plt.tight_layout()
    plt.show()
    print(f"Tổng cộng có {len(misclassified_samples)} ảnh bị nhận diện sai.")
else:
    print("Tuyệt vời! Không tìm thấy ảnh nào bị nhận diện sai.")

In [ ]:
total_images = len(y_true_all) if 'y_true_all' in locals() else len(y_true)
detected_images = len(y_true) # Sau khi đã dùng continue
missing_count = 906 - 850 # Dựa trên kết quả thực tế của bạn

print(f"--- THỐNG KÊ KHẢ NĂNG NHẬN DIỆN ---")
print(f"Tổng số ảnh trong tập test: 906")
print(f"Số ảnh YOLO tìm thấy vật thể: 850")
print(f"Số ảnh YOLO bỏ lỡ (không detect): {missing_count}")
print(f"Tỷ lệ bỏ lỡ (Missing Rate): {(missing_count/906)*100:.2f}%")

if missing_count > 0:
    print("\n=> Lời khuyên: Khi dùng 'continue', báo cáo của bạn chỉ phản ánh chất lượng phân loại class, chứ chưa phản ánh khả năng phát hiện vật thể.")

In [ ]:
import os
import cv2
import torch
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-9/cnn_dataset/test'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best3.pt'

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}

# --- 3. Đánh giá YOLO-Only (Tính cả mẫu không detect được) ---
y_true_yolo = []
y_pred_yolo = []

print(f"--- Đang đánh giá YOLO ONLY (906 ảnh) ---")
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            best_box = boxes[boxes.conf.argmax()]
            cls_yolo = int(best_box.cls[0])
            final_pred = name_to_idx.get(yolo_names[cls_yolo], -1)
        else:
            final_pred = -1 # Không detect được

        y_true_yolo.append(label_idx)
        y_pred_yolo.append(final_pred)

print("\n[BÁO CÁO YOLO ONLY]")
print(classification_report(y_true_yolo, y_pred_yolo, labels=list(range(len(class_names))), target_names=class_names, zero_division=0))

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torchvision import transforms, models
from ultralytics import YOLO
from sklearn.metrics import classification_report

# --- 1. Load CNN Model ---
CNN_MODEL_PATH = '/content/drive/MyDrive/best_convnext_research2.pth'
def load_cnn(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)
    state_dict = torch.load(path, map_location=DEVICE)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE).eval()

cnn_model = load_cnn(CNN_MODEL_PATH, 7)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Đánh giá Fusion (YOLO + CNN) ---
y_true_fusion = []
y_pred_fusion = []

print(f"--- Đang đánh giá FUSION (YOLO + ConvNeXt) (906 ảnh) ---")
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            best_box = boxes[boxes.conf.argmax()]
            conf = float(best_box.conf[0])
            # Nếu YOLO tin cậy cao (>=0.8) dùng YOLO
            if conf >= 0.8:
                final_pred = name_to_idx.get(yolo_names[int(best_box.cls[0])], -1)
            else:
                # YOLO tin cậy thấp -> Dùng CNN crop
                x1, y1, x2, y2 = map(int, best_box.xyxy[0])
                crop = frame[y1:y2, x1:x2]
                img_input = transform(crop if crop.size > 0 else frame).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    final_pred = torch.max(cnn_model(img_input), 1)[1].item()
        else:
            # YOLO không thấy gì -> Dùng CNN trên cả ảnh
            img_input = transform(frame).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                final_pred = torch.max(cnn_model(img_input), 1)[1].item()

        y_true_fusion.append(label_idx)
        y_pred_fusion.append(final_pred)

print("\n[BÁO CÁO FUSION (YOLO + CNN)]")
print(classification_report(y_true_fusion, y_pred_fusion, target_names=class_names, zero_division=0))

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# --- 1. Cấu hình kiểm tra (lặp lại từ cell 673lPPaVygD_) ---
TEST_DIR = '/content/nhận-diện-rác-điện-tử-9/cnn_dataset/test'
MODEL_PATH = '/content/drive/MyDrive/best_convnext_research.pth'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
NUM_CLASSES = 7

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. Khởi tạo Model & Load Weights (lặp lại từ cell 673lPPaVygD_) ---
def load_research_model(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)

    state_dict = torch.load(path, map_location=DEVICE)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model.to(DEVICE)

# --- 3. Load Dữ liệu và chuẩn bị để lưu đường dẫn ảnh ---
test_set = datasets.ImageFolder(TEST_DIR, test_transform)
class_names = test_set.classes

# Cần một custom dataset để lấy đường dẫn ảnh
class ImageFolderWithPaths(datasets.ImageFolder):
    def __getitem__(self, index):
        original_tuple = super().__getitem__(index)
        path = self.imgs[index][0]
        return (original_tuple + (path,))

test_set_with_paths = ImageFolderWithPaths(TEST_DIR, test_transform)
test_loader = DataLoader(test_set_with_paths, batch_size=32, shuffle=False)

model = load_research_model(MODEL_PATH, NUM_CLASSES)
model.eval()

# --- 4. Chạy Inference và lưu kết quả cùng đường dẫn ảnh ---
all_preds = []
all_labels = []
all_img_paths = []

print(f"--- Đang đánh giá mô hình: {MODEL_PATH} và thu thập ảnh lỗi ---")
with torch.no_grad():
    for inputs, labels, paths in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_img_paths.extend(paths)

# --- 5. Xác định và hiển thị các ảnh bị nhận diện sai ---
misclassified_images = []
for i in range(len(all_labels)):
    if all_labels[i] != all_preds[i]:
        misclassified_images.append({
            'path': all_img_paths[i],
            'true_label': class_names[all_labels[i]],
            'predicted_label': class_names[all_preds[i]]
        })

print(f"\nTổng số ảnh bị nhận diện sai: {len(misclassified_images)}")

if len(misclassified_images) > 0:
    print("\n--- Các ảnh bị nhận diện sai ---")
    # Hiển thị tối đa 10 ảnh bị sai để dễ xem
    num_display = min(10, len(misclassified_images))
    fig, axes = plt.subplots(num_display, 1, figsize=(10, num_display * 5))
    if num_display == 1: # Fix for single subplot case
        axes = [axes]

    for i, misclass_info in enumerate(misclassified_images[:num_display]):
        img_path = misclass_info['path']
        true_label = misclass_info['true_label']
        predicted_label = misclass_info['predicted_label']

        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Chuyển BGR sang RGB cho matplotlib
            axes[i].imshow(img)
            axes[i].set_title(f"Thực tế: {true_label} | Dự đoán: {predicted_label}")
            axes[i].axis('off')
        else:
            axes[i].set_title(f"Không thể đọc ảnh: {img_path}")
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy ảnh nào bị nhận diện sai.")

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-7/cnn_dataset/test'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best2.pt'
CNN_MODEL_PATH = '/content/drive/MyDrive/best_convnext_research.pth'
IMG_SIZE = 224
NUM_CLASSES = 7

# --- 2. Load Models ---
yolo_model = YOLO(YOLO_MODEL_PATH)

def load_cnn_model(path, num_classes):
    model = models.convnext_tiny(weights=None)
    n_inputs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(n_inputs, num_classes)
    state_dict = torch.load(path, map_location=DEVICE)
    new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    model.to(DEVICE)
    model.eval()
    return model

cnn_model = load_cnn_model(CNN_MODEL_PATH, NUM_CLASSES)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Đánh giá Fusion ---
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}
yolo_names = yolo_model.names

y_true = []
y_pred = []

print(f"--- Đang đánh giá FUSION (YOLO + CNN) ---")
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            best_box = boxes[boxes.conf.argmax()]
            conf = float(best_box.conf[0])
            cls_yolo = int(best_box.cls[0])
            yolo_mapped_idx = name_to_idx.get(yolo_names[cls_yolo], -1)

            if conf >= 0.7 and yolo_mapped_idx != -1:
                final_pred = yolo_mapped_idx
            else:
                x1, y1, x2, y2 = map(int, best_box.xyxy[0])
                crop = frame[y1:y2, x1:x2]
                img_input = transform(crop if crop.size > 0 else frame).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    final_pred = torch.max(cnn_model(img_input), 1)[1].item()
        else:
            img_input = transform(frame).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                final_pred = torch.max(cnn_model(img_input), 1)[1].item()

        y_true.append(label_idx)
        y_pred.append(final_pred)

print("\n[BÁO CÁO FUSION]")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Fusion (YOLO + CNN) Confusion Matrix')
plt.show()

In [ ]:
import os
import cv2
import torch
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-7/cnn_dataset/test'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best2.pt'

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}

# --- 3. Đánh giá YOLO-Only ---
y_true = []
y_pred = []

print(f"--- Đang đánh giá YOLO ONLY ---")
for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            best_box = boxes[boxes.conf.argmax()]
            cls_yolo = int(best_box.cls[0])
            final_pred = name_to_idx.get(yolo_names[cls_yolo], -1)
        else:
            final_pred = -1

        y_true.append(label_idx)
        y_pred.append(final_pred)

print("\n[BÁO CÁO YOLO ONLY]")
print(classification_report(y_true, y_pred, labels=list(range(len(class_names))), target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=class_names, yticklabels=class_names)
plt.title('YOLO Only Confusion Matrix')
plt.show()

In [ ]:
import os
import cv2
import torch
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Cấu hình ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = '/content/nhận-diện-rác-điện-tử-7/cnn_dataset/test'
YOLO_MODEL_PATH = '/content/drive/MyDrive/best2.pt'
NUM_CLASSES = 7

# --- 2. Load YOLO ---
yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_names = yolo_model.names

# --- 3. Chạy Evaluation ---
class_names = sorted(os.listdir(TEST_DIR))
name_to_idx = {name: i for i, name in enumerate(class_names)}

y_true = []
y_pred = []

print(f"--- Bắt đầu đánh giá YOLO trên: {len(class_names)} lớp ---")

for label_idx, class_name in enumerate(class_names):
    class_path = os.path.join(TEST_DIR, class_name)
    if not os.path.isdir(class_path): continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue

        # YOLO Inference
        results = yolo_model(frame, verbose=False, device=DEVICE)
        boxes = results[0].boxes

        if len(boxes) > 0:
            # Lấy box có confidence cao nhất
            best_box_idx = boxes.conf.argmax()
            cls_yolo = int(boxes[best_box_idx].cls[0])
            yolo_label_name = yolo_names[cls_yolo]

            # Map kết quả YOLO về index của class_names
            final_pred = name_to_idx.get(yolo_label_name, -1)
        else:
            # Nếu YOLO không detect được gì, gán nhãn -1 (Không nhận diện được)
            final_pred = -1

        y_true.append(label_idx)
        y_pred.append(final_pred)

# --- 4. Kết quả ---
print("\n[BÁO CÁO KẾT QUẢ YOLO ONLY]")
# Lưu ý: labels=range(NUM_CLASSES) để bỏ qua các mẫu YOLO không detect được (-1) trong báo cáo nếu cần
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Dự đoán (YOLO)')
plt.ylabel('Thực tế')
plt.title('YOLO Only Confusion Matrix')
plt.show()

In [ ]:
import pandas as pd

# Lấy dữ liệu từ biến cm đã có trong kernel
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

print("--- Ma trận nhầm lẫn (Số lượng mẫu) ---")
display(cm_df)

# Tính toán tỷ lệ phần trăm nhận diện đúng cho mỗi lớp
print("\n--- Tỷ lệ nhận diện đúng từng lớp (Recall %) ---")
for i, class_name in enumerate(class_names):
    accuracy = 100 * cm[i, i] / cm[i].sum()
    print(f"{class_name:15}: {accuracy:.2f}%")

# Tìm các cặp hay bị nhầm lẫn nhất
print("\n--- Các cặp nhầm lẫn đáng chú ý (Top 3) ---")
errors = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            errors.append((class_names[i], class_names[j], cm[i, j]))

errors.sort(key=lambda x: x[2], reverse=True)
for actual, pred, count in errors[:3]:
    print(f"Thực tế là '{actual}' nhưng đoán nhầm thành '{pred}': {count} mẫu")

# kết nối gg colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')